# 🚀 Multi-Label Classification: EfficientNet-B0 + CBAM + ASL (Kaggle T4)

Complete pipeline for ablation study on COCO 2017 dataset with Kaggle GPU T4

## Cell 1: Install Dependencies

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q

## Cell 3: Clone Repository từ GitHub

**Đổi `REPO_URL`** thành URL repo thực trên GitHub trước khi chạy.
Sau khi clone, code sẽ nằm tại `/kaggle/working/ECAAL/`.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'  # ← đổi nếu URL khác
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    print(f'Repo đã tồn tại — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print(f'Cloning {REPO_URL} ...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone thất bại! Kiểm tra REPO_URL.')

# Verify
required = [
    'src/train.py', 'src/losses.py', 'src/models.py',
    'src/dataset.py', 'src/evaluate.py', 'src/cbam.py', 'src/utils.py',
    'configs/exp_A_resnet_bce.yaml', 'configs/exp_B_resnet_asl.yaml',
    'configs/exp_C_efficientnet_cbam_asl.yaml', 'configs/exp_D_resnet_focal.yaml',
    'configs/exp_E_resnet_cbam_asl.yaml', 'configs/exp_F_efficientnet_asl.yaml'
]
all_ok = True
for rel in required:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"  {'OK' if ok else 'MISSING'} {rel}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Một số file thiếu — clone chưa đầy đủ!')
print('\nRepository OK!')

## Cell 4: Setup Working Directory

Tạo các thư mục cần thiết cho outputs và data.

In [ ]:
os.chdir('/kaggle/working')
os.makedirs('/kaggle/working/data/coco_subset', exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
print(f"Working dir: {os.getcwd()}")
print("✅ Directory structure created")

## Cell 4: Copy Code Files to Working Directory

In actual usage, copy the following files from your repo:
- src/losses.py
- src/cbam.py
- src/models.py
- src/dataset.py
- src/evaluate.py
- src/utils.py
- src/train.py
- configs/*.yaml

For demonstration, we'll use a simpler approach here.

In [ ]:
# Or if using git:
# !git clone https://github.com/YOUR_USERNAME/multilabel-cbam-asl.git /kaggle/working/repo
# %cd /kaggle/working/repo

# For now, set working directory
os.chdir('/kaggle/working')
print(f"Working dir: {os.getcwd()}")

## Cell 5: Dataset Preparation

**Before running this:**
1. Go to Notebook → **+ Add Data**
2. Search for `"coco-2017-dataset"` by `awsaf49`
3. Add it to mount at `/kaggle/input/coco-2017-dataset/`

In [ ]:
# Verify COCO dataset is mounted
coco_root = '/kaggle/input/coco-2017-dataset'
if os.path.exists(coco_root):
    train_ann = os.path.join(coco_root, 'annotations', 'instances_train2017.json')
    val_ann = os.path.join(coco_root, 'annotations', 'instances_val2017.json')
    print(f"✅ COCO 2017 dataset found")
    print(f"   Train annotations: {os.path.exists(train_ann)}")
    print(f"   Val annotations: {os.path.exists(val_ann)}")
else:
    print(f"❌ COCO dataset not found at {coco_root}")
    print("   Please add the dataset via Kaggle's Add Data interface")

## Cell 6: Create COCO 2017 Subset

Creates 16k train + 4k val images for faster training

In [ ]:
import json
import random
import numpy as np
from collections import defaultdict

def create_coco_subset(coco_root: str, output_dir: str,
                       num_train: int = 16000, num_val: int = 4000,
                       seed: int = 42):
    """Tạo stratified subset của COCO 2017."""
    random.seed(seed)
    np.random.seed(seed)

    for split, n in [('train', num_train), ('val', num_val)]:
        ann_file = os.path.join(coco_root, 'annotations',
                                f'instances_{split}2017.json')
        with open(ann_file) as f:
            coco = json.load(f)

        img_cats: dict = defaultdict(list)
        for ann in coco['annotations']:
            img_cats[ann['image_id']].append(ann['category_id'])

        all_ids = list(img_cats.keys())
        all_ids.sort(key=lambda i: -len(set(img_cats[i])))

        step = max(1, len(all_ids) // n)
        selected = all_ids[::step][:n]
        
        if len(selected) < n:
            remaining = list(set(all_ids) - set(selected))
            random.shuffle(remaining)
            selected += remaining[:n - len(selected)]

        os.makedirs(output_dir, exist_ok=True)
        out_path = os.path.join(output_dir, f'subset_{split}_ids.json')
        with open(out_path, 'w') as f:
            json.dump(selected[:n], f)
        print(f"[Subset] COCO 2017 {split}: {len(selected[:n])} ids → {out_path}")

# Create subset
create_coco_subset(
    coco_root='/kaggle/input/coco-2017-dataset',
    output_dir='/kaggle/working/data/coco_subset',
    num_train=16000,
    num_val=4000
)

## Cell 7: Train Experiment A - ResNet50 + BCE

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ECAAL/src')

# Run Exp A
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_A_resnet_bce.yaml

## Cell 8: Train Experiment B - ResNet50 + ASL

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_B_resnet_asl.yaml

## Cell 9: Train Experiment D - ResNet50 + Focal Loss

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_D_resnet_focal.yaml

## Cell 10: Train Experiment C - EfficientNet-B0 + CBAM + ASL

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_C_efficientnet_cbam_asl.yaml

## Cell 10.1: Train Experiment E - ResNet50 + CBAM + ASL
Phân tích CBAM trên backbone mạnh.

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_E_resnet_cbam_asl.yaml

## Cell 10.2: Train Experiment F - EfficientNet-B0 + ASL
Phân tích hiệu năng chỉ có ASL trên backbone nhỏ.

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_F_efficientnet_asl.yaml

## Cell 11: Evaluate Results & Plot Curves

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

base = '/kaggle/working/outputs'
exps = {
    'A: ResNet50+BCE':     'exp_A_resnet_bce/log.json',
    'D: ResNet50+Focal':   'exp_D_resnet_focal/log.json',
    'B: ResNet50+ASL':     'exp_B_resnet_asl/log.json',
    'C: EffNet+CBAM+ASL':  'exp_C_efficientnet_cbam_asl/log.json',
    'E: ResNet50+CBAM+ASL': 'exp_E_resnet_cbam_asl/log.json',
    'F: EffNet+ASL':        'exp_F_efficientnet_asl/log.json',
}

rows = []
logs = {}
for name, rel in exps.items():
    path = os.path.join(base, rel)
    if not os.path.exists(path):
        print(f"⚠️  {name} log not found: {path}")
        continue
    records = json.load(open(path))
    logs[name] = records
    best = max(records, key=lambda r: r.get('mAP', 0))
    rows.append({
        'Experiment': name,
        'mAP': f"{best['mAP']:.4f}",
        'Macro F1': f"{best['macro_f1']:.4f}",
        'Best Epoch': best['epoch']
    })

print("\n" + "="*60)
print("ABLATION STUDY RESULTS")
print("="*60)
print(pd.DataFrame(rows).to_string(index=False))
print("="*60 + "\n")

## Cell 12: Plot Training Curves

In [ ]:
if logs:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6', '#1abc9c']
    
    for (name, recs), c in zip(logs.items(), colors):
        ep = [r['epoch'] for r in recs]
        ax1.plot(ep, [r['mAP'] for r in recs], marker='o', label=name, color=c, linewidth=2)
        ax2.plot(ep, [r['macro_f1'] for r in recs], marker='s', label=name, color=c, linewidth=2)

    for ax, title, ylabel in [(ax1, 'mAP vs Epoch', 'mAP'),
                               (ax2, 'Macro F1 vs Epoch', 'Macro F1')]:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(fontsize=9, loc='best')
        ax.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(os.path.join(base, 'ablation_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Curves saved to {os.path.join(base, 'ablation_curves.png')}")
else:
    print("⚠️  No log files found. Run training cells first.")

## Cell 13: Summary Statistics

In [ ]:
if rows:
    print("\n📊 SUMMARY:")
    print(f"  Total Experiments: {len(rows)}")
    best_exp = max(rows, key=lambda x: float(x['mAP']))
    print(f"  Best mAP: {best_exp['Experiment']} ({best_exp['mAP']})")
    print(f"\n  All results have been saved to: {base}")
    print(f"    - Logs: {base}/*/log.json")
    print(f"    - Best models: {base}/*/best.pth")
    print(f"    - Curves: {base}/ablation_curves.png")
else:
    print("No experiments completed yet.")